# Part 2: Preprocessing

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## Part 2.1: Preprocessing

In [7]:
# Load dataset
train = pd.read_csv(r"C:\Users\Hzaab\Desktop\MLSD project\data\raw\train.csv")

train = train.drop(columns=["Unnamed: 0"])

# Log transform large numerical columns
log_cols = ["#follows", "#followers", "description length", "#posts"]
for col in log_cols:
    train[col] = np.log1p(train[col])

In [8]:
categorical_cols = ["profile pic", "name==username", "external URL", "private"]
categorical_cols = [col for col in categorical_cols if col in train.columns]

numeric_columns = [col for col in train.columns if col not in categorical_cols and col != "fake"]


In [9]:
# Split features and target
X = train.drop(columns=["fake"])
y = train["fake"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=10,)


In [10]:
# Preprocessing pipelines
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_columns),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)


In [11]:
# Apply preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)



In [12]:
# Get transformed feature names
num_features = numeric_columns
cat_features = preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)

all_features = np.concatenate([num_features, cat_features])


In [13]:
# Convert to DataFrames
train_processed = pd.DataFrame(X_train_processed, columns=all_features)
val_processed = pd.DataFrame(X_val_processed, columns=all_features)

# Add target back
train_processed["fake"] = y_train.reset_index(drop=True)
val_processed["fake"] = y_val.reset_index(drop=True)


In [15]:
# Save as parquet
train_processed.to_parquet(r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\train_processed.parquet", index=False)

val_processed.to_parquet(r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\val_processed.parquet", index=False)

print("Done")

Done


In [18]:
train_check = pd.read_parquet(r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\train_processed.parquet")

val_check = pd.read_parquet(r"C:\Users\Hzaab\Desktop\MLSD project\data\preprocessed\val_processed.parquet")

print("Train:")
print(train_check.head(0))

print("\nValidation:")
print(val_check.head(0))

Train:
Empty DataFrame
Columns: [nums/length username, fullname words, nums/length fullname, description length, #posts, #followers, #follows, profile pic_0.0, profile pic_1.0, name==username_0.0, name==username_1.0, external URL_0.0, external URL_1.0, private_0.0, private_1.0, fake]
Index: []

Validation:
Empty DataFrame
Columns: [nums/length username, fullname words, nums/length fullname, description length, #posts, #followers, #follows, profile pic_0.0, profile pic_1.0, name==username_0.0, name==username_1.0, external URL_0.0, external URL_1.0, private_0.0, private_1.0, fake]
Index: []
